# Analyze emotion features

Run the setup cell first. Review model, SAE, data paths, and run settings before starting. See the repository README for setup and pipeline order.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


In [ ]:
import os
import traceback

from sae_emotion.visual_utils import run_publishable_feature_analysis

# ============================================================
# Config
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-1b-it",
    # "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

SKIP_MODELS = {
    "google/gemma-3-1b-pt",
    "google/gemma-3-1b-it",
}

BASE_DATA_DIR = "data"
CONTINUE_ON_ERROR = True

SHOW = True
RUN_SEMANTIC_ANALYSIS = False

# ============================================================
# Helpers
# ============================================================

def get_sae_sizes_for_model(model_name: str):
    """
    Gemma-2 and Llama do not use small/big SAE size.
    Gemma-3 4B/12B use both small and big.
    """
    if model_name.startswith("google/gemma-2"):
        return [None]

    if model_name.startswith("meta-llama/"):
        return [None]

    if model_name.startswith("google/gemma-3"):
        return ["small", "big"]

    return [None]


def get_feature_dir(model_name: str, sae_size):
    model_slug = model_name.replace("/", "-")

    if sae_size is None:
        return f"{BASE_DATA_DIR}/{model_slug}-feature-act"

    return f"{BASE_DATA_DIR}/{model_slug}-feature-act-{sae_size}"


def get_paths(model_name: str, sae_size):
    feature_dir = get_feature_dir(model_name, sae_size)

    input_path = (
        f"{feature_dir}/auto_selected_unique_features_with_affected_emotions.csv"
    )
    output_dir = f"{feature_dir}/emotion_feature_analysis"

    return {
        "feature_dir": feature_dir,
        "input_path": input_path,
        "output_dir": output_dir,
    }


# ============================================================
# Main
# ============================================================

all_results = {}
run_log = []

for model_name in MODELS:
    if model_name in SKIP_MODELS:
        print("\n===================================================")
        print("Skipping model:", model_name)
        print("===================================================")

        run_log.append({
            "model_name": model_name,
            "sae_size": None,
            "status": "skipped_model",
            "error": "",
        })
        continue

    for sae_size in get_sae_sizes_for_model(model_name):
        paths = get_paths(model_name, sae_size)

        print("\n===================================================")
        print("Running publishable feature analysis")
        print("===================================================")
        print("MODEL_NAME:", model_name)
        print("SAE_SIZE:", sae_size)
        print("input_path:", paths["input_path"])
        print("output_dir:", paths["output_dir"])

        if not os.path.exists(paths["input_path"]):
            print("\nSkipping because input file does not exist:")
            print(paths["input_path"])

            run_log.append({
                "model_name": model_name,
                "sae_size": sae_size,
                "status": "skipped_missing_input",
                "error": paths["input_path"],
            })
            continue

        try:
            results = run_publishable_feature_analysis(
                input_path=paths["input_path"],
                output_dir=paths["output_dir"],
                show=SHOW,
                run_semantic_analysis=RUN_SEMANTIC_ANALYSIS,
            )

            model_key = model_name.replace("/", "-")
            if sae_size is not None:
                model_key = f"{model_key}-{sae_size}"

            all_results[model_key] = results

            paper_summary = results["paper_summary"]
            concentration = results["concentration"]

            print("\nFinished analysis")
            print("MODEL_NAME:", model_name)
            print("SAE_SIZE:", sae_size)
            print("paper_summary shape:", paper_summary.shape)
            print("concentration shape:", concentration.shape)

            run_log.append({
                "model_name": model_name,
                "sae_size": sae_size,
                "status": "done",
                "error": "",
            })

        except Exception as e:
            print("\nRun failed")
            print("MODEL_NAME:", model_name)
            print("SAE_SIZE:", sae_size)
            print("Error:", repr(e))
            traceback.print_exc()

            run_log.append({
                "model_name": model_name,
                "sae_size": sae_size,
                "status": "failed",
                "error": repr(e),
            })

            if not CONTINUE_ON_ERROR:
                raise

print("\n===================================================")
print("All publishable feature analyses finished or skipped.")
print("===================================================")

for item in run_log:
    print(item)

In [ ]:
import os
import pandas as pd
import numpy as np


# ============================================================
# Config
# ============================================================

MODEL_RUNS = [
    {"model_name": "google/gemma-2-2b", "sae_size": "canonical"},
    {"model_name": "google/gemma-2-9b", "sae_size": "canonical"},

    {"model_name": "google/gemma-3-4b-pt", "sae_size": "big"},
    {"model_name": "google/gemma-3-4b-pt", "sae_size": "small"},

    {"model_name": "google/gemma-3-4b-it", "sae_size": "big"},
    {"model_name": "google/gemma-3-4b-it", "sae_size": "small"},

    {"model_name": "google/gemma-3-12b-pt", "sae_size": "big"},
    {"model_name": "google/gemma-3-12b-pt", "sae_size": "small"},

    {"model_name": "google/gemma-3-12b-it", "sae_size": "big"},
    {"model_name": "google/gemma-3-12b-it", "sae_size": "small"},

    {"model_name": "meta-llama/Llama-3.1-8B", "sae_size": "llamascope"},
]

EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

BASE_DATA_DIR = "data"

OUTPUT_CSV = "data/feature_summary_by_model_emotion.csv"


# ============================================================
# Helpers
# ============================================================

def model_safe_name(model_name: str) -> str:
    return model_name.replace("/", "-")


def display_model_name(model_name: str, sae_size: str) -> str:
    name_map = {
        "google/gemma-2-2b": "Gemma-2-2B",
        "google/gemma-2-9b": "Gemma-2-9B",
        "google/gemma-3-4b-pt": "Gemma-3-4B-PT",
        "google/gemma-3-4b-it": "Gemma-3-4B-IT",
        "google/gemma-3-12b-pt": "Gemma-3-12B-PT",
        "google/gemma-3-12b-it": "Gemma-3-12B-IT",
        "meta-llama/Llama-3.1-8B": "Llama-3.1-8B",
    }

    base = name_map.get(model_name, model_name)

    if "gemma-3" in model_name:
        return f"{base}-{sae_size}"

    return base


def get_feature_csv_path(model_name: str, sae_size: str) -> str:
    safe_name = model_safe_name(model_name)

    if "gemma-3" in model_name:
        return os.path.join(
            BASE_DATA_DIR,
            f"{safe_name}-feature-act-{sae_size}",
            "auto_selected_unique_features_with_affected_emotions.csv",
        )

    return os.path.join(
        BASE_DATA_DIR,
        f"{safe_name}-feature-act",
        "auto_selected_unique_features_with_affected_emotions.csv",
    )


def parse_affected_emotions(x):
    if pd.isna(x):
        return []

    x = str(x).lower().strip()

    if x == "":
        return []

    for ch in ["[", "]", "'", '"']:
        x = x.replace(ch, "")

    return [p.strip() for p in x.split(",") if p.strip() != ""]


def summarize_one_run(model_name: str, sae_size: str):
    csv_path = get_feature_csv_path(model_name, sae_size)

    if not os.path.exists(csv_path):
        print(f"[Missing] {csv_path}")
        return None

    df = pd.read_csv(csv_path)

    required_cols = ["layer_idx", "feature_idx", "affected_emotions"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} missing columns: {missing}")

    for emotion in EMOTIONS:
        logit_col = f"logit_{emotion}"
        if logit_col not in df.columns:
            raise ValueError(f"{csv_path} missing column: {logit_col}")

    df["layer_idx"] = df["layer_idx"].astype(int)
    df["feature_idx"] = df["feature_idx"].astype(int)

    # Defensive deduplication
    df = df.drop_duplicates(subset=["layer_idx", "feature_idx"]).copy()

    df["_affected_list"] = df["affected_emotions"].apply(parse_affected_emotions)

    row = {
        "model": display_model_name(model_name, sae_size),
        "model_name": model_name,
        "sae_size": sae_size,
        "total_features": int(len(df)),
    }

    for emotion in EMOTIONS:
        mask = df["_affected_list"].apply(lambda xs: emotion in xs)
        sub = df[mask].copy()

        row[f"{emotion}_features"] = int(len(sub))

    for emotion in EMOTIONS:
        logit_col = f"logit_{emotion}"
        mask = df["_affected_list"].apply(lambda xs: emotion in xs)
        sub = df[mask].copy()

        if len(sub) == 0:
            row[f"{emotion}_avg_logit_weight_per_feature"] = np.nan
        else:
            row[f"{emotion}_avg_logit_weight_per_feature"] = float(
                pd.to_numeric(sub[logit_col], errors="coerce").mean()
            )

    row["csv_path"] = csv_path

    return row


# ============================================================
# Build table
# ============================================================

rows = []

for run in MODEL_RUNS:
    row = summarize_one_run(
        model_name=run["model_name"],
        sae_size=run["sae_size"],
    )

    if row is not None:
        rows.append(row)

summary_df = pd.DataFrame(rows)


# ============================================================
# Add average row
# ============================================================

feature_count_cols = ["total_features"] + [
    f"{emotion}_features" for emotion in EMOTIONS
]

logit_weight_cols = [
    f"{emotion}_avg_logit_weight_per_feature" for emotion in EMOTIONS
]

avg_row = {
    "model": "Average",
    "model_name": "Average",
    "sae_size": "",
    "csv_path": "",
}

for col in feature_count_cols:
    avg_row[col] = summary_df[col].mean()

for col in logit_weight_cols:
    avg_row[col] = summary_df[col].mean()

summary_df = pd.concat(
    [summary_df, pd.DataFrame([avg_row])],
    ignore_index=True,
)


# ============================================================
# Reorder columns
# ============================================================

ordered_cols = [
    "model",
    "sae_size",
    "total_features",
]

for emotion in EMOTIONS:
    ordered_cols.append(f"{emotion}_features")

for emotion in EMOTIONS:
    ordered_cols.append(f"{emotion}_avg_logit_weight_per_feature")

ordered_cols += [
    "model_name",
    "csv_path",
]

summary_df = summary_df[ordered_cols]


# ============================================================
# Round numeric columns
# ============================================================

for col in summary_df.columns:
    if col not in {"model", "model_name", "sae_size", "csv_path"}:
        summary_df[col] = pd.to_numeric(summary_df[col], errors="coerce").round(4)

# Optional: feature counts as 1 decimal for average row consistency
for col in feature_count_cols:
    summary_df[col] = pd.to_numeric(summary_df[col], errors="coerce").round(2)


# ============================================================
# Save
# ============================================================

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
summary_df.to_csv(OUTPUT_CSV, index=False)

print("Saved to:", OUTPUT_CSV)
display(summary_df)

In [ ]:
from sae_emotion.visual_utils import run_all_model_publishable_feature_aggregate_plot

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-1b-it",
    # "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

results = run_all_model_publishable_feature_aggregate_plot(
    models=MODELS,
    skip_models={
        "google/gemma-3-1b-pt",
        "google/gemma-3-1b-it",
    },
    base_data_dir="data",
    output_dir="data/all_model_emotion_feature_analysis",
    show=True,
)

paper_summary_all = results["paper_summary_all"]
concentration_all = results["concentration_all"]
overlap_all = results["overlap_all"]
figures = results["figures"]

## Activation Patching: ablation

In [ ]:
import os
import traceback

from sae_emotion.visual_utils import plot_ablation_summary

# ============================================================
# Config
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-1b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

SKIP_MODELS = {
    "google/gemma-3-1b-pt",
    "google/gemma-3-1b-it",
}

BASE_DATA_DIR = "data"
SHOW = True
CONTINUE_ON_ERROR = True


# ============================================================
# Helpers
# ============================================================

def get_sae_sizes_for_model(model_name: str):
    """
    Gemma-2 and Llama use old folders without SAE size suffix.
    Gemma-3 4B/12B use both small and big.
    """
    if model_name.startswith("google/gemma-2"):
        return [None]

    if model_name.startswith("meta-llama/"):
        return [None]

    if model_name.startswith("google/gemma-3"):
        return ["small", "big"]

    return [None]


def get_feature_dir(model_name: str, sae_size):
    model_slug = model_name.replace("/", "-")

    if sae_size is None:
        return f"{BASE_DATA_DIR}/{model_slug}-feature-act"

    return f"{BASE_DATA_DIR}/{model_slug}-feature-act-{sae_size}"


# ============================================================
# Main
# ============================================================

all_ablation_plot_results = {}
run_log = []

for model_name in MODELS:
    if model_name in SKIP_MODELS:
        print("\n===================================================")
        print("Skipping model:", model_name)
        print("===================================================")

        run_log.append({
            "model_name": model_name,
            "sae_size": None,
            "status": "skipped_model",
            "error": "",
        })
        continue

    for sae_size in get_sae_sizes_for_model(model_name):
        feature_dir = get_feature_dir(model_name, sae_size)

        summary_path = (
            f"{feature_dir}/"
            "per_layer_per_emotion_feature_ablation/"
            "per_layer_per_emotion_ablation_summary.csv"
        )

        output_dir = (
            f"{feature_dir}/"
            "per_layer_per_emotion_feature_ablation/plots"
        )

        print("\n===================================================")
        print("Running ablation visualization")
        print("===================================================")
        print("MODEL_NAME:", model_name)
        print("SAE_SIZE:", sae_size)
        print("summary_path:", summary_path)
        print("output_dir:", output_dir)

        if not os.path.exists(summary_path):
            print("Skipping because summary file does not exist:", summary_path)

            run_log.append({
                "model_name": model_name,
                "sae_size": sae_size,
                "status": "skipped_missing_summary",
                "error": summary_path,
            })
            continue

        try:
            ablation_results = plot_ablation_summary(
                summary_path=summary_path,
                output_dir=output_dir,
                show=SHOW,
            )

            model_key = model_name.replace("/", "-")
            if sae_size is not None:
                model_key = f"{model_key}-{sae_size}"

            all_ablation_plot_results[model_key] = ablation_results

            print("Finished ablation visualization:", model_key)

            run_log.append({
                "model_name": model_name,
                "sae_size": sae_size,
                "status": "done",
                "error": "",
            })

        except Exception as e:
            print("Run failed")
            print("MODEL_NAME:", model_name)
            print("SAE_SIZE:", sae_size)
            print("Error:", repr(e))
            traceback.print_exc()

            run_log.append({
                "model_name": model_name,
                "sae_size": sae_size,
                "status": "failed",
                "error": repr(e),
            })

            if not CONTINUE_ON_ERROR:
                raise


print("\n===================================================")
print("All ablation visualizations finished or skipped.")
print("===================================================")

for item in run_log:
    print(item)

In [ ]:
from sae_emotion.visual_utils import plot_all_model_ablation_summary

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-1b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

results = plot_all_model_ablation_summary(
    models=MODELS,
    skip_models={
        "google/gemma-3-1b-pt",
        "google/gemma-3-1b-it",
    },
    base_data_dir="data",
    output_dir="data/all_model_ablation_summary",
    show=True,
)

ablation_all = results["df"]
run_log = results["run_log"]
figures = results["figures"]

In [ ]:
from sae_emotion.visual_utils import plot_swap_summary, plot_all_model_swap_summary

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-1b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

SKIP_MODELS = {
    "google/gemma-3-1b-pt",
    "google/gemma-3-1b-it",
}

BASE_DATA_DIR = "data"


def get_sae_sizes_for_model(model_name):
    if model_name.startswith("google/gemma-2"):
        return [None]
    if model_name.startswith("meta-llama/"):
        return [None]
    if model_name.startswith("google/gemma-3"):
        return ["small", "big"]
    return [None]


def get_feature_dir(model_name, sae_size):
    model_slug = model_name.replace("/", "-")
    if sae_size is None:
        return f"{BASE_DATA_DIR}/{model_slug}-feature-act"
    return f"{BASE_DATA_DIR}/{model_slug}-feature-act-{sae_size}"


# ============================================================
# Plot each model and display figures
# ============================================================

for model_name in MODELS:
    if model_name in SKIP_MODELS:
        print("Skipping:", model_name)
        continue

    for sae_size in get_sae_sizes_for_model(model_name):
        feature_dir = get_feature_dir(model_name, sae_size)

        summary_path = (
            f"{feature_dir}/cross_emotion_layer_feature_swap/"
            f"cross_emotion_layer_feature_swap_summary.csv"
        )

        output_dir = (
            f"{feature_dir}/cross_emotion_layer_feature_swap/plots"
        )

        print("\n===================================================")
        print("Plotting:", model_name, sae_size)
        print("summary_path:", summary_path)
        print("output_dir:", output_dir)
        print("===================================================")

        try:
            plot_swap_summary(
                summary_path=summary_path,
                output_dir=output_dir,
                show=True,   # display plots
                save=True,
                pair_curves=[
                    ("joy", "fear"),
                    ("joy", "disgust"),
                    ("disgust", "anger"),
                    ("anger", "disgust"),
                    ("surprise", "joy"),
                    ("fear", "surprise"),
                ],
            )
        except FileNotFoundError:
            print("Missing summary file:", summary_path)
        except Exception as e:
            print("Failed:", model_name, sae_size, repr(e))


# ============================================================
# Plot aggregated results and display figures
# ============================================================

results = plot_all_model_swap_summary(
    models=MODELS,
    base_data_dir="data",
    output_dir="data/all_model_cross_emotion_layer_feature_swap",
    skip_models=SKIP_MODELS,
    show=True,   # display aggregate plots
    save=True,
)

print("Done.")

## Subgroup analysis

In [ ]:
from sae_emotion.subtype_utils import (
    SubtypeAnalysisConfig,
    run_subtype_discovery,
    plot_subtype_analysis,
)

MODEL_NAME = "google/gemma-3-12b-pt"
SAE_SIZE = "big"  # None, "small", or "big"
TARGET_EMOTION = "disgust"

model_slug = MODEL_NAME.replace("/", "-")

if SAE_SIZE is None:
    feature_act_dir = f"data/{model_slug}-feature-act"
    act_path = f"data/sae_acts_combined_zeroshot_{model_slug}.npy"
else:
    feature_act_dir = f"data/{model_slug}-feature-act-{SAE_SIZE}"
    act_path = f"data/sae_acts_combined_zeroshot_{model_slug}_{SAE_SIZE}.npy"

config = SubtypeAnalysisConfig(
    model_name=MODEL_NAME,
    pred_path=f"data/combined_zeroshot_preds_{model_slug}.csv",
    act_path=act_path,
    feature_path=(
        f"{feature_act_dir}/"
        "auto_selected_unique_features_with_affected_emotions.csv"
    ),
    output_dir=(
        f"{feature_act_dir}/"
        "distributed_emotion_subroute_analysis"
    ),
    target_emotion=TARGET_EMOTION,
    run_causal_ablation=False,
)

run_subtype_discovery(config)
plot_subtype_analysis(config)

disgust is distributed
disgust swaps diffuse into multiple negative emotions
disgust lacks a compact attractor

The model internally separates different disgust-related semantic contexts into partially distinct sparse activation substructures.

## co-activation

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# Config
# ============================================================

PRED_PATH = "data/combined_zeroshot_preds_google-gemma-2-2b.csv"

ACT_PATH = "data/sae_acts_combined_zeroshot_google-gemma-2-2b.npy"

FEATURE_PATH = (
    "data/google-gemma-2-2b-feature-act/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

OUT_DIR = (
    "data/google-gemma-2-2b-feature-act/"
    "emotion_feature_coactivation"
)

os.makedirs(OUT_DIR, exist_ok=True)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

# Feature active threshold
ACTIVE_THRESHOLD = 0.0

# Ensemble active rule:
# An emotion ensemble is active on an instance if at least this many selected features are active.
MIN_ACTIVE_FEATURES = 1

# Use only correctly predicted instances?
USE_CORRECT_ONLY = True


# ============================================================
# Helpers
# ============================================================

def parse_emotions(x):
    if pd.isna(x):
        return set()
    return set([v.strip() for v in str(x).split(",") if v.strip() != ""])


def phi_corr(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)

    n11 = np.logical_and(a, b).sum()
    n10 = np.logical_and(a, ~b).sum()
    n01 = np.logical_and(~a, b).sum()
    n00 = np.logical_and(~a, ~b).sum()

    denom = np.sqrt(
        (n11 + n10)
        * (n01 + n00)
        * (n11 + n01)
        * (n10 + n00)
    )

    if denom == 0:
        return np.nan

    return ((n11 * n00) - (n10 * n01)) / denom


def safe_corr(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    if np.std(x) < 1e-8 or np.std(y) < 1e-8:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])


def savefig(name):
    path = os.path.join(OUT_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)
    plt.show()


# ============================================================
# Load data
# ============================================================

pred_df = pd.read_csv(PRED_PATH)
acts = np.load(ACT_PATH, mmap_mode="r")
feature_df = pd.read_csv(FEATURE_PATH)

print("Prediction shape:", pred_df.shape)
print("Activation shape:", acts.shape)
print("Feature shape:", feature_df.shape)

required_pred_cols = ["input_text", "emotion", "pred_emotion"]
missing = [c for c in required_pred_cols if c not in pred_df.columns]
if missing:
    raise ValueError(f"Missing prediction columns: {missing}")

required_feature_cols = ["layer_idx", "feature_idx", "affected_emotions"]
missing = [c for c in required_feature_cols if c not in feature_df.columns]
if missing:
    raise ValueError(f"Missing feature columns: {missing}")

pred_df["emotion"] = pred_df["emotion"].astype(str).str.lower().str.strip()
pred_df["pred_emotion"] = pred_df["pred_emotion"].astype(str).str.lower().str.strip()

pred_df = pred_df[pred_df["emotion"].isin(EMOTIONS)].copy()
pred_df = pred_df[pred_df["pred_emotion"].isin(EMOTIONS)].copy()

if USE_CORRECT_ONLY:
    pred_df = pred_df[pred_df["emotion"] == pred_df["pred_emotion"]].copy()

pred_df = pred_df.reset_index(drop=False).rename(columns={"index": "original_instance_idx"})

print("Used instances:", pred_df.shape)
print(pred_df["emotion"].value_counts())

instance_indices = pred_df["original_instance_idx"].astype(int).to_numpy()

feature_df["layer_idx"] = feature_df["layer_idx"].astype(int)
feature_df["feature_idx"] = feature_df["feature_idx"].astype(int)
feature_df["affected_emotion_set"] = feature_df["affected_emotions"].apply(parse_emotions)

feature_df = feature_df.drop_duplicates(subset=["layer_idx", "feature_idx"]).copy()

# ============================================================
# Build emotion ensemble activations
# ============================================================

ensemble_rows = []

emotion_feature_map = {}

for emotion in EMOTIONS:
    sub = feature_df[
        feature_df["affected_emotion_set"].apply(lambda s: emotion in s)
    ].copy()

    feature_list = list(
        zip(
            sub["layer_idx"].astype(int).tolist(),
            sub["feature_idx"].astype(int).tolist(),
        )
    )

    emotion_feature_map[emotion] = feature_list

    print(f"{emotion}: {len(feature_list)} features")

    if len(feature_list) == 0:
        continue

    feature_acts = []

    for layer_idx, feature_idx in feature_list:
        vec = np.asarray(
            acts[layer_idx, instance_indices, feature_idx],
            dtype=np.float32,
        )
        feature_acts.append(vec)

    # shape: n_features × n_instances
    feature_acts = np.stack(feature_acts, axis=0)

    active_binary = feature_acts > ACTIVE_THRESHOLD

    ensemble_mean = feature_acts.mean(axis=0)
    ensemble_max = feature_acts.max(axis=0)
    active_count = active_binary.sum(axis=0)
    active_rate = active_binary.mean(axis=0)

    ensemble_active = active_count >= MIN_ACTIVE_FEATURES

    for local_i, original_idx in enumerate(instance_indices):
        ensemble_rows.append(
            {
                "original_instance_idx": int(original_idx),
                "true_emotion": pred_df.loc[local_i, "emotion"],
                "pred_emotion": pred_df.loc[local_i, "pred_emotion"],
                "emotion_feature_group": emotion,
                "n_features": int(len(feature_list)),
                "ensemble_mean_activation": float(ensemble_mean[local_i]),
                "ensemble_max_activation": float(ensemble_max[local_i]),
                "active_feature_count": int(active_count[local_i]),
                "active_feature_rate": float(active_rate[local_i]),
                "ensemble_active": bool(ensemble_active[local_i]),
            }
        )

ensemble_df = pd.DataFrame(ensemble_rows)

ensemble_out = os.path.join(OUT_DIR, "emotion_ensemble_activation_by_instance.csv")
ensemble_df.to_csv(ensemble_out, index=False)
print("Saved:", ensemble_out)


# ============================================================
# Wide table: one row per instance, columns per emotion group
# ============================================================

wide_active = ensemble_df.pivot(
    index="original_instance_idx",
    columns="emotion_feature_group",
    values="ensemble_active",
).reindex(columns=EMOTIONS)

wide_mean = ensemble_df.pivot(
    index="original_instance_idx",
    columns="emotion_feature_group",
    values="ensemble_mean_activation",
).reindex(columns=EMOTIONS)

wide_count = ensemble_df.pivot(
    index="original_instance_idx",
    columns="emotion_feature_group",
    values="active_feature_count",
).reindex(columns=EMOTIONS)

meta = pred_df.set_index("original_instance_idx")[["emotion", "pred_emotion", "input_text"]]

wide_active = wide_active.join(meta)
wide_mean = wide_mean.join(meta)
wide_count = wide_count.join(meta)

wide_active_path = os.path.join(OUT_DIR, "wide_emotion_ensemble_active.csv")
wide_mean_path = os.path.join(OUT_DIR, "wide_emotion_ensemble_mean_activation.csv")
wide_count_path = os.path.join(OUT_DIR, "wide_emotion_active_feature_count.csv")

wide_active.to_csv(wide_active_path)
wide_mean.to_csv(wide_mean_path)
wide_count.to_csv(wide_count_path)

print("Saved:", wide_active_path)
print("Saved:", wide_mean_path)
print("Saved:", wide_count_path)


# ============================================================
# Pairwise activation-strength metrics
# ============================================================

pair_rows = []

def cosine_sim(x, y):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    denom = np.linalg.norm(x) * np.linalg.norm(y)
    if denom < 1e-8:
        return np.nan

    return float(np.dot(x, y) / denom)


def normalized_dot(x, y):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    return float(np.mean(x * y))


def mean_strength_when_a_high(a, b, q=0.75):
    threshold = np.quantile(a, q)
    idx = a >= threshold

    if idx.sum() == 0:
        return np.nan, np.nan, np.nan

    return (
        float(np.mean(a[idx])),
        float(np.mean(b[idx])),
        float(np.mean(b[idx] - a[idx])),
    )


for emotion_a in EMOTIONS:
    for emotion_b in EMOTIONS:
        if emotion_a == emotion_b:
            continue

        a_mean = wide_mean[emotion_a].astype(float).to_numpy()
        b_mean = wide_mean[emotion_b].astype(float).to_numpy()

        a_count = wide_count[emotion_a].astype(float).to_numpy()
        b_count = wide_count[emotion_b].astype(float).to_numpy()

        strength_corr = safe_corr(a_mean, b_mean)
        count_corr = safe_corr(a_count, b_count)
        cos = cosine_sim(a_mean, b_mean)
        ndot = normalized_dot(a_mean, b_mean)

        mean_a = float(np.mean(a_mean))
        mean_b = float(np.mean(b_mean))

        mean_abs_a = float(np.mean(np.abs(a_mean)))
        mean_abs_b = float(np.mean(np.abs(b_mean)))

        mean_margin_b_minus_a = float(np.mean(b_mean - a_mean))
        median_margin_b_minus_a = float(np.median(b_mean - a_mean))

        frac_b_stronger = float(np.mean(b_mean > a_mean))
        frac_a_stronger = float(np.mean(a_mean > b_mean))

        a_top_mean, b_when_a_top_mean, margin_when_a_top = mean_strength_when_a_high(
            a_mean,
            b_mean,
            q=0.75,
        )

        pair_rows.append(
            {
                "emotion_a": emotion_a,
                "emotion_b": emotion_b,
                "n_instances": int(len(a_mean)),

                "mean_activation_a": mean_a,
                "mean_activation_b": mean_b,
                "mean_abs_activation_a": mean_abs_a,
                "mean_abs_activation_b": mean_abs_b,

                "strength_corr": strength_corr,
                "active_count_corr": count_corr,
                "cosine_similarity": cos,
                "normalized_dot": ndot,

                "mean_margin_b_minus_a": mean_margin_b_minus_a,
                "median_margin_b_minus_a": median_margin_b_minus_a,
                "frac_b_stronger_than_a": frac_b_stronger,
                "frac_a_stronger_than_b": frac_a_stronger,

                "a_top25_mean_activation": a_top_mean,
                "b_mean_activation_when_a_top25": b_when_a_top_mean,
                "margin_b_minus_a_when_a_top25": margin_when_a_top,
            }
        )

pair_df = pd.DataFrame(pair_rows)

pair_out = os.path.join(
    OUT_DIR,
    "pairwise_emotion_feature_activation_strength.csv"
)
pair_df.to_csv(pair_out, index=False)
print("Saved:", pair_out)

print("\nTop pairs by activation strength correlation:")
print(
    pair_df
    .sort_values("strength_corr", ascending=False)
    .head(20)
    [
        [
            "emotion_a",
            "emotion_b",
            "strength_corr",
            "cosine_similarity",
            "normalized_dot",
            "mean_margin_b_minus_a",
            "frac_b_stronger_than_a",
        ]
    ]
)


# ============================================================
# Pairwise activation-strength metrics conditioned on true emotion
# ============================================================

condition_rows = []

for condition_emotion in EMOTIONS:
    idx = wide_mean["emotion"].astype(str).to_numpy() == condition_emotion

    if idx.sum() == 0:
        continue

    for emotion_a in EMOTIONS:
        for emotion_b in EMOTIONS:
            if emotion_a == emotion_b:
                continue

            a_mean = wide_mean.loc[idx, emotion_a].astype(float).to_numpy()
            b_mean = wide_mean.loc[idx, emotion_b].astype(float).to_numpy()

            a_count = wide_count.loc[idx, emotion_a].astype(float).to_numpy()
            b_count = wide_count.loc[idx, emotion_b].astype(float).to_numpy()

            strength_corr = safe_corr(a_mean, b_mean)
            count_corr = safe_corr(a_count, b_count)
            cos = cosine_sim(a_mean, b_mean)
            ndot = normalized_dot(a_mean, b_mean)

            mean_a = float(np.mean(a_mean))
            mean_b = float(np.mean(b_mean))

            mean_margin_b_minus_a = float(np.mean(b_mean - a_mean))
            median_margin_b_minus_a = float(np.median(b_mean - a_mean))

            frac_b_stronger = float(np.mean(b_mean > a_mean))
            frac_a_stronger = float(np.mean(a_mean > b_mean))

            condition_rows.append(
                {
                    "condition_true_emotion": condition_emotion,
                    "emotion_a": emotion_a,
                    "emotion_b": emotion_b,
                    "n_instances": int(len(a_mean)),

                    "mean_activation_a": mean_a,
                    "mean_activation_b": mean_b,

                    "strength_corr": strength_corr,
                    "active_count_corr": count_corr,
                    "cosine_similarity": cos,
                    "normalized_dot": ndot,

                    "mean_margin_b_minus_a": mean_margin_b_minus_a,
                    "median_margin_b_minus_a": median_margin_b_minus_a,
                    "frac_b_stronger_than_a": frac_b_stronger,
                    "frac_a_stronger_than_b": frac_a_stronger,
                }
            )

condition_df = pd.DataFrame(condition_rows)

condition_out = os.path.join(
    OUT_DIR,
    "pairwise_emotion_feature_activation_strength_conditioned_on_true_emotion.csv"
)
condition_df.to_csv(condition_out, index=False)
print("Saved:", condition_out)


# ============================================================
# Heatmaps
# ============================================================

def plot_pair_heatmap(pair_df, value_col, title, filename):
    pivot = pair_df.pivot(
        index="emotion_a",
        columns="emotion_b",
        values=value_col,
    ).reindex(index=EMOTIONS, columns=EMOTIONS)

    plt.figure(figsize=(8, 6))
    im = plt.imshow(pivot.values, aspect="auto")
    plt.colorbar(im, label=value_col)

    plt.xticks(np.arange(len(EMOTIONS)), EMOTIONS, rotation=45, ha="right")
    plt.yticks(np.arange(len(EMOTIONS)), EMOTIONS)

    plt.xlabel("Emotion B")
    plt.ylabel("Emotion A")
    plt.title(title)

    for i in range(len(EMOTIONS)):
        for j in range(len(EMOTIONS)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                plt.text(
                    j,
                    i,
                    f"{val:.2f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                )

    savefig(filename)


plot_pair_heatmap(
    pair_df,
    value_col="strength_corr",
    title="Correlation Between Emotion Ensemble Activation Strengths",
    filename="heatmap_activation_strength_correlation.png",
)

plot_pair_heatmap(
    pair_df,
    value_col="cosine_similarity",
    title="Cosine Similarity Between Emotion Ensemble Activation Strengths",
    filename="heatmap_activation_strength_cosine_similarity.png",
)

plot_pair_heatmap(
    pair_df,
    value_col="mean_margin_b_minus_a",
    title="Mean Activation Margin: Emotion B minus Emotion A",
    filename="heatmap_mean_activation_margin_b_minus_a.png",
)

plot_pair_heatmap(
    pair_df,
    value_col="frac_b_stronger_than_a",
    title="Fraction of Instances Where Emotion B Is Stronger Than Emotion A",
    filename="heatmap_fraction_b_stronger_than_a.png",
)


# ============================================================
# Specific pair plots: activation-strength margin
# ============================================================

PAIR_LIST = [
    ("surprise", "joy"),
    ("joy", "surprise"),
    ("disgust", "anger"),
    ("anger", "disgust"),
]

for emotion_a, emotion_b in PAIR_LIST:
    sub = condition_df[
        (condition_df["emotion_a"] == emotion_a)
        & (condition_df["emotion_b"] == emotion_b)
    ].copy()

    sub = sub.set_index("condition_true_emotion").reindex(EMOTIONS).reset_index()

    plt.figure(figsize=(9, 5))

    plt.bar(
        sub["condition_true_emotion"],
        sub["mean_margin_b_minus_a"],
    )

    plt.axhline(0, color="black", linewidth=1)

    plt.xlabel("Conditioned on True Emotion")
    plt.ylabel(f"Mean activation margin: {emotion_b} - {emotion_a}")
    plt.title(f"Activation Strength Competition: {emotion_a} vs {emotion_b}")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.3)

    savefig(f"bar_conditioned_activation_margin_{emotion_a}_vs_{emotion_b}.png")


# ============================================================
# Inspect examples where two emotion groups co-activate
# ============================================================

def save_coactive_examples(emotion_a, emotion_b, top_n=100):
    a_active = wide_active[emotion_a].astype(bool)
    b_active = wide_active[emotion_b].astype(bool)

    sub = wide_active[a_active & b_active].copy()
    sub_mean = wide_mean.loc[sub.index].copy()
    sub_count = wide_count.loc[sub.index].copy()

    out = pd.DataFrame(
        {
            "original_instance_idx": sub.index.astype(int),
            "true_emotion": sub["emotion"].values,
            "pred_emotion": sub["pred_emotion"].values,
            "input_text": sub["input_text"].values,
            f"{emotion_a}_mean_activation": sub_mean[emotion_a].values,
            f"{emotion_b}_mean_activation": sub_mean[emotion_b].values,
            f"{emotion_a}_active_feature_count": sub_count[emotion_a].values,
            f"{emotion_b}_active_feature_count": sub_count[emotion_b].values,
        }
    )

    out["joint_score"] = (
        out[f"{emotion_a}_mean_activation"]
        + out[f"{emotion_b}_mean_activation"]
    )

    out = out.sort_values("joint_score", ascending=False).head(top_n)

    path = os.path.join(
        OUT_DIR,
        f"top_coactive_examples_{emotion_a}_{emotion_b}.csv"
    )
    out.to_csv(path, index=False)
    print("Saved:", path)

    return out


save_coactive_examples("surprise", "joy", top_n=100)
save_coactive_examples("disgust", "anger", top_n=100)

print("\nDone.")
print("Output directory:", OUT_DIR)